# Text2Cypher Toolkit: Kaggle Pipeline

Runs the complete pipeline for **one** model on a Kaggle T4: bootstrap, download, diagnostics, LoRA fine-tuning, inference, evaluation and GGUF export. This is the Kaggle counterpart of `google-colab.ipynb`; both share the same `src/` code, configs and Hub checkpoint repo, so a run can move between platforms.

### One-time setup
1. Verify your Kaggle account by phone (required for GPU and internet).
2. **Settings > Accelerator > GPU T4 x2.** Do not pick P100 (Pascal): recent PyTorch builds are dropping it and the configs are tuned for a T4.
3. **Settings > Internet > On.**
4. **Add-ons > Secrets:** add `HF_TOKEN` (write access) and attach it to this notebook.

### Running
- Edit the values in the **Pipeline configuration** cell.
- For unattended runs use **Save Version > Save & Run All (Commit)**. Interactive sessions time out when idle.
- Only one of the two T4s is used. With two visible GPUs, HF `Trainer` would wrap the 4-bit model in `DataParallel`, which is fragile with bitsandbytes and would double the effective batch size relative to the Colab runs.

### Time budget
- Kaggle GPU sessions are capped at about 12 h (`SESSION_BUDGET_MINUTES`). The training wall-clock limit is the smaller of `TIME_LIMIT_MINUTES` and the session time left after `RESERVE_MINUTES`, which covers model loading, tokenization (both happen before the training clock starts), the final checkpoint upload, and prediction/evaluation.
- If training pauses at the limit, its checkpoint is on the Hub and the remaining cells are skipped, so the version still completes and its logs are saved. Commit a new version to resume. Once a final adapter exists on the Hub, later runs skip training.

### Logs and results
- Repo, models and artifacts live in `SCRATCH_DIR` (not saved). Logs and small result files go to `OUTPUT_DIR/text2cypher-logs/<model>-<variant>/<session>/`, which Kaggle saves as the version's output.
- Logs are also mirrored to the Hub under `logs/<model>-<variant>/<session>/` every `LOG_UPLOAD_INTERVAL_S` seconds and at the end of each step; results go to `results/<model>-<variant>/<session>/`.

### Notifications
- With `NTFY_TOPIC` set, push alerts go to `<NTFY_SERVER>/<NTFY_TOPIC>` when the pipeline starts, training starts, a step fails, training pauses, the pipeline finishes, or the kernel ends before finishing.
- A hard kill at the session cap may not deliver an alert. ntfy.sh topics are public; only status text and log tails are sent, never tokens.

In [ ]:
# 1. Pipeline configuration (edit these values, then Save Version > Save & Run All)
import time

PIPELINE_T0 = time.time()  # Session time budget is measured from here

MODEL = "gemma-4-e2b"            # gemma-4-e2b | qwen3.5-2b | granite-4.2-3b
SCHEMA_VARIANT = "enhanced"      # enhanced | pruned (Granite: enhanced only)
HUB_REPO = "2l3c34o1/text2cypher-checkpoints"
GITHUB_REPO = "https://github.com/Cxmrykk/capstone.git"

SESSION_BUDGET_MINUTES = 720     # Kaggle GPU session cap (12 h)
RESERVE_MINUTES = 75             # Setup before the training clock starts + upload + predict/eval
TIME_LIMIT_MINUTES = 600         # Upper bound on training wall-clock time per session
SKIP_TRAINING_IF_FINAL_EXISTS = True

PREDICT_LIMIT = 100
RUN_EXECUTION_EVAL = True
RUN_EXPORT = True
UPLOAD_GGUF = True

LOG_UPLOAD_INTERVAL_S = 300
NTFY_TOPIC = "cxmrykk"           # Blank disables notifications
NTFY_SERVER = "https://ntfy.sh"

SCRATCH_DIR = "/kaggle/temp"     # Repo, models, artifacts (not saved; falls back to /tmp)
OUTPUT_DIR = "/kaggle/working"   # Logs and small results (saved with the notebook version)
GPU_INDEX = "0"                  # Pin one T4 for every child process

CONFIGS = {
    ("gemma-4-e2b", "enhanced"): "configs/gemma-4-e2b.yaml",
    ("gemma-4-e2b", "pruned"): "configs/gemma-4-e2b-pruned.yaml",
    ("qwen3.5-2b", "enhanced"): "configs/qwen3.5-2b.yaml",
    ("qwen3.5-2b", "pruned"): "configs/qwen3.5-2b-pruned.yaml",
    ("granite-4.2-3b", "enhanced"): "configs/granite-4.2-3b.yaml",
}
DOWNLOAD_TARGETS = {"gemma-4-e2b": "gemma", "qwen3.5-2b": "qwen", "granite-4.2-3b": "granite"}

if (MODEL, SCHEMA_VARIANT) not in CONFIGS:
    print(f"Schema variant '{SCHEMA_VARIANT}' is not defined for {MODEL} "
          "(Granite is the enhanced-schema control); using 'enhanced'.")
    SCHEMA_VARIANT = "enhanced"

CONFIG = CONFIGS[(MODEL, SCHEMA_VARIANT)]
DOWNLOAD_TARGET = DOWNLOAD_TARGETS[MODEL]
# Only the Qwen configs use Unsloth.
INSTALL_UNSLOTH = MODEL == "qwen3.5-2b"

print(f"Model            : {MODEL}")
print(f"Schema variant   : {SCHEMA_VARIANT}")
print(f"Config           : {CONFIG}")
print(f"Download target  : {DOWNLOAD_TARGET}")
print(f"Install Unsloth  : {INSTALL_UNSLOTH}")
print(f"Hub repo         : {HUB_REPO}")
print(f"Session budget   : {SESSION_BUDGET_MINUTES} min (reserve {RESERVE_MINUTES} min, "
      f"training cap {TIME_LIMIT_MINUTES} min)")
print(f"Notifications    : {NTFY_SERVER.rstrip('/')}/{NTFY_TOPIC.strip()}" if NTFY_TOPIC.strip()
      else "Notifications    : disabled")

In [ ]:
# 2. Platform checks, logging helpers, notifications and secrets
import atexit
import codecs
import datetime
import json
import os
import shutil
import signal
import subprocess
import sys
import threading
import urllib.request
from pathlib import Path

CR, LF = chr(13), chr(10)

# --------------------------------------------------------------------------- #
# Platform checks (fail fast, before anything slow runs)
# --------------------------------------------------------------------------- #
ON_KAGGLE = Path("/kaggle").is_dir() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if not ON_KAGGLE:
    raise RuntimeError("This notebook targets Kaggle. On Google Colab use google-colab.ipynb.")
RUN_TYPE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "unknown")  # Interactive | Batch (commit)

# Pin one GPU for every child process. With GPU T4 x2, HF Trainer would otherwise
# wrap the 4-bit model in nn.DataParallel: fragile with bitsandbytes, and it would
# double the effective batch size relative to the Colab runs.
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_INDEX


def _query_gpus():
    """Lists physical GPUs as 'index, name, memory' (nvidia-smi ignores CUDA_VISIBLE_DEVICES)."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30,
        ).stdout
    except Exception:
        return []
    return [line.strip() for line in out.splitlines() if line.strip()]


GPUS = _query_gpus()
if not GPUS:
    raise RuntimeError("No GPU visible. Settings > Accelerator > GPU T4 x2 "
                       "(requires a phone-verified Kaggle account).")
pinned = [g for g in GPUS if g.split(",")[0].strip() == GPU_INDEX]
if not pinned:
    raise RuntimeError(f"GPU_INDEX={GPU_INDEX} not found. Visible GPUs: {GPUS}")
GPU_NAME = pinned[0].split(",")[1].strip()
if "P100" in GPU_NAME:
    raise RuntimeError(
        "Tesla P100 (Pascal, sm_60) selected. Recent PyTorch CUDA builds are dropping Pascal "
        "support and the configs are tuned for a T4. Switch the accelerator to GPU T4 x2.")
if "T4" not in GPU_NAME:
    print(f"WARNING: expected a Tesla T4, found {GPU_NAME}. Configs are sized for 16 GB fp16.")


def _internet_ok(url="https://huggingface.co", timeout=15):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            return resp.status < 500
    except Exception:
        return False


if not _internet_ok():
    raise RuntimeError("No internet access. Settings > Internet > On "
                       "(requires a phone-verified Kaggle account).")


def _writable_dir(path):
    try:
        p = Path(path)
        p.mkdir(parents=True, exist_ok=True)
        probe = p / ".t2c-write-test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return True
    except Exception:
        return False


def _free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return float("nan")


WORK_ROOT = Path(SCRATCH_DIR) if _writable_dir(SCRATCH_DIR) else Path("/tmp")
REPO_DIR = WORK_ROOT / "capstone"

# --------------------------------------------------------------------------- #
# Session, log locations and secrets
# --------------------------------------------------------------------------- #
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S") + "-kaggle"
RUN_KEY = f"{MODEL}-{SCHEMA_VARIANT}"
LOG_ROOT = Path(OUTPUT_DIR) / "text2cypher-logs"
LOG_DIR = LOG_ROOT / RUN_KEY / SESSION
LOG_DIR.mkdir(parents=True, exist_ok=True)
HUB_LOG_PREFIX = f"logs/{RUN_KEY}/{SESSION}"
HUB_RESULTS_PREFIX = f"results/{RUN_KEY}/{SESSION}"
HUB_LOGS_URL = f"https://huggingface.co/{HUB_REPO}/tree/main/{HUB_LOG_PREFIX}" if HUB_REPO else None

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as exc:
    # Scratch storage is wiped when the session ends, so training without Hub
    # sync would lose all progress. Refuse to start instead.
    raise RuntimeError(
        f"Could not read the 'HF_TOKEN' Kaggle secret ({type(exc).__name__}). "
        "Add-ons > Secrets: add HF_TOKEN (write access) and attach it to this notebook."
    ) from exc

os.environ["T2C_HUB_REPO"] = HUB_REPO
# Reduce PyTorch memory fragmentation on the 16 GB T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Child Python processes flush output immediately, so logs stay current.
os.environ["PYTHONUNBUFFERED"] = "1"

STATUS = {
    "platform": "kaggle",
    "run_type": RUN_TYPE,
    "gpu": GPU_NAME,
    "model": MODEL,
    "schema_variant": SCHEMA_VARIANT,
    "config": CONFIG,
    "session": SESSION,
    "run_name": None,
    "current_step": None,
    # None while running; then completed | failed | paused | interrupted | runtime-ended.
    "outcome": None,
    "steps": [],
}
# Set when training pauses (time limit) or no session time is left. Later cells
# check it and skip, so a committed version still completes and saves its output.
HALTED = None
_hub_state = {"repo_ready": False}
_hub_lock = threading.Lock()


# --------------------------------------------------------------------------- #
# ntfy push notifications
# --------------------------------------------------------------------------- #
NTFY_URL = f"{NTFY_SERVER.rstrip('/')}/{NTFY_TOPIC.strip()}" if NTFY_TOPIC.strip() else None
NTFY_PREFIX = f"T2C {RUN_KEY} (Kaggle)"


def notify(title, message, priority="default", tags=(), click=None):
    """Best-effort push notification via ntfy. Never raises.

    The topic is public: anyone who knows its name can read it, so only
    status text and log tails are sent, never tokens.
    """
    if not NTFY_URL:
        return False
    headers = {
        # HTTP header values must be latin-1; keep the title plain ASCII.
        "Title": title.encode("ascii", "replace").decode("ascii"),
        "Priority": str(priority),
    }
    if tags:
        headers["Tags"] = ",".join(tags)
    if click or HUB_LOGS_URL:
        headers["Click"] = click or HUB_LOGS_URL
    # ntfy turns bodies over 4096 bytes into attachments; keep the end of the text.
    body = message if len(message) <= 3000 else "..." + message[-3000:]
    req = urllib.request.Request(NTFY_URL, data=body.encode("utf-8"),
                                 headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            resp.read()
        return True
    except Exception as exc:
        print(f"[ntfy] Notification failed: {exc}", file=sys.stderr)
        return False


def _log_tail(path, n=15):
    """Returns the last n lines of a step log (empty string if unreadable)."""
    try:
        lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    except Exception:
        return ""
    return LF.join(lines[-n:])


def hub_upload(local_path, path_in_repo, blocking=False):
    """Best-effort upload of a file or folder to the Hub repo. Never raises."""
    if not HUB_REPO or not os.environ.get("HF_TOKEN"):
        return
    local_path = Path(local_path)
    if not local_path.exists():
        return

    def _do():
        try:
            from huggingface_hub import HfApi
            api = HfApi(token=os.environ["HF_TOKEN"])
            with _hub_lock:
                if not _hub_state["repo_ready"]:
                    api.create_repo(HUB_REPO, repo_type="model", private=True, exist_ok=True)
                    _hub_state["repo_ready"] = True
                if local_path.is_dir():
                    api.upload_folder(folder_path=str(local_path), path_in_repo=path_in_repo,
                                      repo_id=HUB_REPO, repo_type="model",
                                      commit_message=f"[{RUN_KEY}] {path_in_repo}")
                else:
                    api.upload_file(path_or_fileobj=str(local_path), path_in_repo=path_in_repo,
                                    repo_id=HUB_REPO, repo_type="model",
                                    commit_message=f"[{RUN_KEY}] {path_in_repo}")
        except Exception as exc:
            print(f"[log-sync] Upload of {local_path.name} failed: {exc}", file=sys.stderr)

    if blocking:
        _do()
    else:
        threading.Thread(target=_do, daemon=True).start()


def save_status():
    path = LOG_DIR / "pipeline_status.json"
    path.write_text(json.dumps(STATUS, indent=2), encoding="utf-8")
    hub_upload(path, f"{HUB_LOG_PREFIX}/pipeline_status.json")


def archive_results(paths):
    """Copies result files next to the logs (saved output) and uploads them to the Hub."""
    dest = LOG_DIR / "results"
    dest.mkdir(parents=True, exist_ok=True)
    for p in map(Path, paths):
        if not p.exists():
            continue
        if p.is_dir():
            shutil.copytree(p, dest / p.name, dirs_exist_ok=True)
        else:
            shutil.copy2(p, dest / p.name)
        hub_upload(p, f"{HUB_RESULTS_PREFIX}/{p.name}", blocking=True)


def _clean_lines(buf, text):
    """Splits streamed output into finished lines. Only the last carriage-return
    segment of each line is kept, so tqdm redraws do not bloat the log file."""
    text = buf + text.replace(CR + LF, LF)
    parts = text.split(LF)
    lines = [p.split(CR)[-1] for p in parts[:-1]]
    return lines, parts[-1].split(CR)[-1]


TRAIN_EXIT_HINTS = {
    2: "The config fingerprint differs from the remote checkpoint. Change run_name, or delete the old run on the Hub.",
    3: "Training loss became non-finite (fp16 overflow); no final adapter was saved.",
    4: "Training reached its time budget and checkpointed to the Hub. Commit a new version "
       "(Save Version > Save & Run All): training resumes automatically.",
}


def run_step(name, cmd, allow_fail=False, env=None, cwd=None):
    """Runs a shell command, streaming its output to the cell and to a log file."""
    index = len(STATUS["steps"]) + 1
    log_path = LOG_DIR / f"{index:02d}-{name}.log"
    hub_path = f"{HUB_LOG_PREFIX}/{log_path.name}"
    workdir = cwd or (str(REPO_DIR) if REPO_DIR.exists() else str(WORK_ROOT))
    started = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    STATUS["current_step"] = name
    print("=" * 72)
    print(f"[{index:02d}] {name}: {cmd}")
    print(f"     log: {log_path}")
    print("=" * 72, flush=True)

    t0 = time.time()
    last_sync = t0
    buf = ""
    interrupted = False
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    with open(log_path, "w", encoding="utf-8") as fh:
        fh.write(f"$ {cmd}" + LF + f"# cwd: {workdir}" + LF + f"# started: {started}" + LF + LF)
        fh.flush()
        proc = subprocess.Popen(
            cmd, shell=True, cwd=workdir, env={**os.environ, **(env or {})},
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True,
        )
        try:
            while True:
                chunk = proc.stdout.read1(8192)
                if not chunk:
                    break
                text = decoder.decode(chunk)
                sys.stdout.write(text)
                sys.stdout.flush()
                lines, buf = _clean_lines(buf, text)
                for line in lines:
                    fh.write(line + LF)
                fh.flush()
                if time.time() - last_sync >= LOG_UPLOAD_INTERVAL_S:
                    hub_upload(log_path, hub_path)
                    last_sync = time.time()
            rc = proc.wait()
        except KeyboardInterrupt:
            # Forward the interrupt so training can write its emergency checkpoint.
            interrupted = True
            try:
                os.killpg(proc.pid, signal.SIGINT)
                rc = proc.wait(timeout=120)
            except Exception:
                proc.kill()
                rc = proc.wait()
        if buf:
            fh.write(buf + LF)
        elapsed = time.time() - t0
        fh.write(LF + f"# exit code: {rc}" + LF + f"# elapsed: {elapsed:.1f}s" + LF)

    hub_upload(log_path, hub_path, blocking=True)
    STATUS["current_step"] = None
    STATUS["steps"].append({
        "index": index, "name": name, "cmd": cmd, "exit_code": rc,
        "elapsed_s": round(elapsed, 1), "started": started, "log": str(log_path),
        "interrupted": interrupted,
    })
    save_status()

    outcome = "OK" if rc == 0 else f"FAILED (exit {rc})"
    print(f"[{index:02d}] {name}: {outcome} in {elapsed / 60:.1f} min", flush=True)

    if interrupted:
        STATUS["outcome"] = "interrupted"
        save_status()
        notify(f"{NTFY_PREFIX}: interrupted",
               f"Step '{name}' was stopped by the user after {elapsed / 60:.1f} min. Session {SESSION}.",
               priority="high", tags=["stop_sign"])
        raise KeyboardInterrupt(f"Step '{name}' interrupted by user.")

    if rc != 0 and not allow_fail:
        hint = TRAIN_EXIT_HINTS.get(rc) if name == "train" else None
        tail = _log_tail(log_path)
        paused = name == "train" and rc == 4
        if paused:
            STATUS["outcome"] = "paused"
            notify(f"{NTFY_PREFIX}: paused at time limit", LF.join([
                hint,
                f"Session {SESSION}, trained for {elapsed / 60:.0f} min.",
                "",
                tail,
            ]), priority="high", tags=["hourglass"])
        else:
            STATUS["outcome"] = "failed"
            notify(f"{NTFY_PREFIX}: step '{name}' failed", LF.join([
                f"Exit code {rc} after {elapsed / 60:.1f} min. {hint or ''}".strip(),
                f"Session {SESSION}. Last output:",
                tail,
            ]), priority="urgent", tags=["rotating_light"])
        save_status()
        error = RuntimeError(f"Step '{name}' failed with exit code {rc}. {hint or f'See {log_path}'}")
        # Already reported; stops the cell-error hook from sending a duplicate.
        error.ntfy_sent = True
        # Lets the training cell treat a time-limit pause as a clean halt.
        error.halt = "paused" if paused else None
        raise error
    return rc


# --------------------------------------------------------------------------- #
# Crash / shutdown hooks
# --------------------------------------------------------------------------- #
def _notify_unfinished(reason):
    """Sends one alert if the kernel goes away before the pipeline reached an outcome."""
    if STATUS.get("outcome") is not None:
        return
    STATUS["outcome"] = "runtime-ended"
    step = STATUS.get("current_step")
    where = f"while step '{step}' was running" if step else "between steps"
    done = ", ".join(s["name"] for s in STATUS["steps"]) or "none"
    notify(f"{NTFY_PREFIX}: runtime ended", LF.join([
        f"{reason} {where}, before the pipeline finished.",
        f"Session {SESSION}. Completed steps: {done}.",
        "Hub checkpoints are kept; commit a new version to resume.",
    ]), priority="urgent", tags=["skull"])


def _on_sigterm(signum, frame):
    _notify_unfinished("The Kaggle session was shut down (SIGTERM)")
    prev = globals().get("_NTFY_PREV_SIGTERM")
    if callable(prev):
        prev(signum, frame)
    else:
        signal.signal(signal.SIGTERM, signal.SIG_DFL)
        os.kill(os.getpid(), signal.SIGTERM)


def _on_cell_exception(shell, etype, evalue, tb, tb_offset=None):
    """Alerts on any uncaught cell error that run_step has not already reported."""
    if not getattr(evalue, "ntfy_sent", False):
        STATUS["outcome"] = "failed"
        try:
            save_status()
        except Exception:
            pass
        notify(f"{NTFY_PREFIX}: notebook error",
               f"{etype.__name__}: {evalue}" + LF + f"Session {SESSION}.",
               priority="urgent", tags=["rotating_light"])
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)


# Install the exit hooks once per kernel, even if this cell is re-run.
if not globals().get("_NTFY_HOOKS_INSTALLED"):
    _NTFY_PREV_SIGTERM = signal.getsignal(signal.SIGTERM)
    signal.signal(signal.SIGTERM, _on_sigterm)
    atexit.register(lambda: _notify_unfinished("The notebook kernel exited"))
    _NTFY_HOOKS_INSTALLED = True
get_ipython().set_custom_exc((Exception,), _on_cell_exception)


save_status()
print(f"Platform   : Kaggle ({RUN_TYPE} session)")
print(f"GPUs       : {'; '.join(GPUS)}")
print(f"Pinned GPU : {GPU_INDEX} ({GPU_NAME}) via CUDA_VISIBLE_DEVICES")
print(f"Work root  : {WORK_ROOT} ({_free_gb(WORK_ROOT):.0f} GB free, not saved)")
print(f"Output dir : {OUTPUT_DIR} ({_free_gb(OUTPUT_DIR):.0f} GB free, saved with the version)")
if _free_gb(WORK_ROOT) < 40:
    print("WARNING: under 40 GB free in the work root. Model weights, the merged model and "
          "the f16 GGUF intermediate may not fit; consider RUN_EXPORT = False.")
print(f"Session    : {SESSION}")
print(f"Local logs : {LOG_DIR}")
if HUB_LOGS_URL:
    print(f"Hub logs   : {HUB_LOGS_URL}")
if NTFY_URL:
    print(f"Notify     : {NTFY_URL}")
    notify(f"{NTFY_PREFIX}: pipeline started",
           f"Session {SESSION} started on Kaggle ({RUN_TYPE}, {GPU_NAME}, {CONFIG}).",
           priority="low", tags=["rocket"])

In [ ]:
# 3. Clone or update the repository
if (REPO_DIR / ".git").exists():
    run_step("git-pull", f"git -C {REPO_DIR} pull --ff-only", allow_fail=True, cwd=str(WORK_ROOT))
else:
    run_step("git-clone", f"git clone {GITHUB_REPO} {REPO_DIR}", cwd=str(WORK_ROOT))
os.chdir(REPO_DIR)

import yaml

RUN_NAME = yaml.safe_load(Path(CONFIG).read_text(encoding="utf-8"))["run_name"]
STATUS["run_name"] = RUN_NAME
save_status()

ADAPTER_DIR = f"artifacts/runs/{RUN_NAME}/final"
PRED_PATH = f"artifacts/predictions/{RUN_NAME}-test.jsonl"
REPORT_PATH = f"artifacts/reports/{RUN_NAME}-eval.md"
MERGED_DIR = f"artifacts/merged/{RUN_NAME}"

print(f"Repository  : {REPO_DIR}")
print(f"Run name    : {RUN_NAME}")
print(f"Adapter     : {ADAPTER_DIR}")
print(f"Predictions : {PRED_PATH}")
print(f"Report      : {REPORT_PATH}")

In [ ]:
# 4. Install dependencies (colab_bootstrap.sh is platform-neutral: pip + hf CLI check)
run_step("bootstrap", "bash scripts/colab_bootstrap.sh",
         env={"INSTALL_UNSLOTH": "1" if INSTALL_UNSLOTH else "0"})

In [ ]:
# 5. Download the dataset and the selected model
run_step("download", f"bash download_data.sh {DOWNLOAD_TARGET}")

In [ ]:
# 6. Diagnostics and data preview
run_step("doctor", "python app.py doctor", allow_fail=True)
run_step("data-preview", f"python app.py data preview --config {CONFIG} --index 5", allow_fail=True)
run_step("token-stats", f"python app.py data token-stats --config {CONFIG} --samples 500", allow_fail=True)

In [ ]:
# 7. LoRA fine-tuning (resumes from the Hub automatically)
adapter_marker = Path(ADAPTER_DIR) / "adapter_config.json"

if SKIP_TRAINING_IF_FINAL_EXISTS and not adapter_marker.exists():
    # Exit code 1 simply means no finished adapter exists on the Hub yet.
    run_step("pull-final", f"python app.py checkpoint pull --config {CONFIG} --step final",
             allow_fail=True)

if SKIP_TRAINING_IF_FINAL_EXISTS and adapter_marker.exists():
    print(f"Final adapter found at {ADAPTER_DIR}; skipping training.")
else:
    elapsed_min = (time.time() - PIPELINE_T0) / 60
    available_min = SESSION_BUDGET_MINUTES - elapsed_min - RESERVE_MINUTES
    train_limit = int(min(TIME_LIMIT_MINUTES, available_min))
    print(f"Session time used so far: {elapsed_min:.0f} min. "
          f"Training wall-clock limit: {train_limit} min.")

    if train_limit < 15:
        HALTED = "no-time"
        STATUS["outcome"] = "paused"
        save_status()
        notify(f"{NTFY_PREFIX}: not enough session time",
               f"Only {available_min:.0f} min left for training in session {SESSION}. "
               "Commit a new version to continue.",
               priority="high", tags=["hourglass"])
        print("Not enough session time left to train; later steps will be skipped.")
    else:
        stop_at = (datetime.datetime.now(datetime.timezone.utc)
                   + datetime.timedelta(minutes=train_limit))
        notify(f"{NTFY_PREFIX}: training started",
               f"Run {RUN_NAME}, session {SESSION}, limit {train_limit} min. The training clock "
               f"starts after model loading and tokenization, so if the schedule does not finish "
               f"first, expect a 'paused at time limit' alert roughly 10-30 min after "
               f"{stop_at:%H:%M} UTC. Silence well past that means the session was killed.",
               priority="low", tags=["hourglass_flowing_sand"])
        try:
            run_step("train", f"python app.py train --config {CONFIG} --resume auto "
                              f"--time-limit {train_limit}")
        except RuntimeError as exc:
            if getattr(exc, "halt", None) != "paused":
                raise
            HALTED = "paused"
            print(str(exc))
            print("Later steps will be skipped; this version still completes so its logs are saved.")
        finally:
            run_dir = Path(f"artifacts/runs/{RUN_NAME}")
            archive_results([
                run_dir / "metrics.jsonl",
                run_dir / "run_config.json",
                run_dir / "final" / "run_metadata.json",
            ])

In [ ]:
# 8. Inference on the test split
if HALTED:
    print(f"Pipeline halted ({HALTED}); skipping inference.")
else:
    run_step("predict",
             f"python app.py predict --config {CONFIG} --adapter {ADAPTER_DIR} --split test "
             f"--limit {PREDICT_LIMIT} --out {PRED_PATH} --tag {RUN_NAME}")

    pred = Path(PRED_PATH)
    archive_results([pred, pred.with_suffix(".meta.json")])

In [ ]:
# 9. Evaluation (GLEU, exact match, execution accuracy)
if HALTED:
    print(f"Pipeline halted ({HALTED}); skipping evaluation.")
else:
    eval_flags = "--execute --validate-syntax" if RUN_EXECUTION_EVAL else ""
    run_step("evaluate",
             f"python app.py evaluate --predictions {PRED_PATH} {eval_flags} --out {REPORT_PATH}")

    pred = Path(PRED_PATH)
    report = Path(REPORT_PATH)
    archive_results([
        pred.with_name(pred.stem + ".scored.jsonl"),
        report,
        report.with_suffix(".json"),
    ])

In [ ]:
# 10. Export to GGUF (best-effort)
if HALTED:
    print(f"Pipeline halted ({HALTED}); skipping export.")
elif not RUN_EXPORT:
    print("RUN_EXPORT is disabled; skipping merge and GGUF conversion.")
else:
    ok = run_step("merge", f"python app.py export merge --config {CONFIG} "
                           f"--adapter {ADAPTER_DIR} --out {MERGED_DIR}", allow_fail=True) == 0
    if ok:
        # A CPU build is enough for convert + quantize and compiles far faster than CUDA.
        ok = run_step("build-llama-cpp", "bash scripts/build_llama_cpp.sh",
                      env={"BUILD_CUDA": "0"}, allow_fail=True) == 0
    if ok:
        ok = run_step("gguf", f"python app.py export gguf --merged {MERGED_DIR} --quant Q4_K_M",
                      allow_fail=True) == 0
    if ok:
        for gguf in sorted(Path("artifacts/gguf").glob(f"{RUN_NAME}-*.gguf*")):
            if gguf.suffix == ".json":
                archive_results([gguf])
            elif UPLOAD_GGUF:
                # The GGUF stays in scratch storage (not saved output); the Hub is its only copy.
                print(f"Uploading {gguf.name} ({gguf.stat().st_size / 1e9:.2f} GB) to the Hub...")
                hub_upload(gguf, f"gguf/{RUN_NAME}/{gguf.name}", blocking=True)
    else:
        print("Export did not complete; see the logs above. Training and evaluation results are unaffected.")

In [ ]:
# 11. Summary
from IPython.display import Markdown, display

print(f"Session {SESSION}: {RUN_KEY} (run_name={RUN_NAME}, {RUN_TYPE}, {GPU_NAME})")
for step in STATUS["steps"]:
    print(f"  [{step['index']:02d}] {step['name']:<16} exit={step['exit_code']:<4} "
          f"{step['elapsed_s'] / 60:7.1f} min")
print(f"Total session time: {(time.time() - PIPELINE_T0) / 60:.0f} min")
print(f"Local logs: {LOG_DIR}")
if HUB_REPO:
    print(f"Hub logs  : https://huggingface.co/{HUB_REPO}/tree/main/{HUB_LOG_PREFIX}")
    print(f"Results   : https://huggingface.co/{HUB_REPO}/tree/main/{HUB_RESULTS_PREFIX}")

if HALTED:
    # The pause/no-time alert was already sent; outcome is already 'paused'.
    save_status()
    print(f"Pipeline halted ({HALTED}). Commit a new version to resume training from the Hub.")
else:
    summary_lines = [f"Session {SESSION} finished ({RUN_NAME})."]
    report_json = Path(REPORT_PATH).with_suffix(".json")
    if report_json.exists():
        try:
            overall = json.loads(report_json.read_text(encoding="utf-8"))[0].get("overall", {})
            for key, label in (("n", "N"), ("google_bleu_corpus", "GLEU"),
                               ("exact_match", "Exact match"),
                               ("normalised_exact_match", "Norm. EM"),
                               ("execution_accuracy", "Exec. acc.")):
                if key in overall:
                    summary_lines.append(f"{label}: {overall[key]}")
        except Exception as exc:
            summary_lines.append(f"(could not read evaluation report: {exc})")
    # pull-final exiting 1 only means no adapter was on the Hub yet, so it is not listed.
    soft_failures = [s["name"] for s in STATUS["steps"]
                     if s["exit_code"] != 0 and s["name"] != "pull-final"]
    if soft_failures:
        summary_lines.append(f"Best-effort steps with errors: {', '.join(soft_failures)}")

    STATUS["outcome"] = "completed"
    save_status()
    notify(f"{NTFY_PREFIX}: pipeline finished", LF.join(summary_lines),
           tags=["warning"] if soft_failures else ["white_check_mark"],
           click=f"https://huggingface.co/{HUB_REPO}/tree/main/{HUB_RESULTS_PREFIX}" if HUB_REPO else None)

    if Path(REPORT_PATH).exists():
        display(Markdown(Path(REPORT_PATH).read_text(encoding="utf-8")))